In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv

# Force reload of .env file
env_path = Path('.').parent / '.env'  # Or Path('./.env') depending on notebook location
load_dotenv(dotenv_path=env_path, override=True)

print("Actual Base URL in memory:", os.getenv("DHIS2_BASE_URL"))

Actual Base URL in memory: https://play.im.dhis2.org/stable-2-41-10/api


In [2]:
import importlib
import os
import pandas as pd
import requests
import config


In [3]:
# Force reload config with override
importlib.reload(config)
from config import DHIS2_BASE_URL, DHIS2_PASSWORD, DHIS2_USERNAME


In [4]:
# Set up auth and clean endpoint path
auth = (DHIS2_USERNAME, DHIS2_PASSWORD)
base_url = DHIS2_BASE_URL.rstrip("/")


In [5]:
# Target the dataElements API endpoint
endpoint = f"{base_url}/dataElements.json"

params = {"filter": "name:like:ANC", "fields": "id,name", "paging": "false"}

resp = requests.get(endpoint, params=params, auth=auth)

print(f" Request URL: {resp.url}")
print(f" Status Code: {resp.status_code}\n")

if resp.status_code == 200:
    data = resp.json()
    data_elements = data.get("dataElements", [])
    df_elements = pd.DataFrame(data_elements)
    print(
        f" Success! Found {len(df_elements)} ANC data elements in Demo instance:"
    )
    print(df_elements.head(10))
else:
    print(f" Error {resp.status_code}: {resp.text[:300]}")

 Request URL: https://play.im.dhis2.org/stable-2-41-10/api/dataElements.json?filter=name%3Alike%3AANC&fields=id%2Cname&paging=false
 Status Code: 200

 Success! Found 16 ANC data elements in Demo instance:
                                       name           id
0  Albendazole given at ANC (2nd trimester)  hCVSHjcml9g
1                             ANC 1st visit  fbfJHSPpUQD
2                             ANC 2nd visit  cYeuwXTCPkU
3                             ANC 3rd visit  Jtf34kNZhzP
4                    ANC 4th or more visits  hfdmMSPBgLG
5              Iron Folate given at ANC 3rd  rbkr8PL0rwM
6                    LLITN given at ANC 1st  ybzlGLjWwnK
7                             MCH ANC Visit  g9eOBujte1U
8                       MNCH ANC Attendance  WQSdEl9xUMA
9                      MNCH ANC registrants  iYCGteGcr6Q


In [6]:
# Step 2: pull actual values for a chosen data element + org unit level + period
# Replace DE_UID and OU_UID with real values found above / from the Maintenance app
DE_UID = "fbfJHSPpUQD"
OU_UID = "fdc6uOvgoji"   # e.g. a district/level in the demo's Sierra Leone hierarchy
PERIOD = "LAST_12_MONTHS"                       # adjust to an available period in the demo
 
resp2 = requests.get(
    f"{DHIS2_BASE_URL}/analytics.json",
    params={
        "dimension": [f"dx:{DE_UID}", f"ou:{OU_UID}", f"pe:{PERIOD}"],
    },
    auth=auth,
)
resp2.raise_for_status()
data = resp2.json()
 
rows = data.get("rows", [])
headers = [h["name"] for h in data.get("headers", [])]
dhis2_df = pd.DataFrame(rows, columns=headers)
print(dhis2_df.head())
 
dhis2_df.to_csv("dhis2_indicator_values.csv", index=False)
print("Saved: dhis2_indicator_values.csv")
 

            dx           ou      pe value
0  fbfJHSPpUQD  fdc6uOvgoji  202512  1410
1  fbfJHSPpUQD  fdc6uOvgoji  202601  1523
2  fbfJHSPpUQD  fdc6uOvgoji  202602  1387
3  fbfJHSPpUQD  fdc6uOvgoji  202510  1379
4  fbfJHSPpUQD  fdc6uOvgoji  202607  1599
Saved: dhis2_indicator_values.csv
